<a href="https://colab.research.google.com/github/drhrf/XML_HPA_parser/blob/main/HPA_XML_Parser.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HPA XML Parser

Extract **per-sample** RNA expression (nTPM, sex, age) for tissues, human, pig and mouse
brain regions and immune cells from the [Human Protein Atlas](https://www.proteinatlas.org/)
per-gene XML files.

Run the cells from top to bottom. No file paths need to be typed: genes are downloaded by
Ensembl ID or gene symbol.

## 1. Install

In [ ]:
%pip install -q "hpa-xml-parser @ git+https://github.com/drhrf/XML_HPA_parser"

In [ ]:
import hpa_xml_parser as hxp

print("hpa-xml-parser", hxp.__version__)

## 2. Download one gene

`fetch_gene` accepts an Ensembl ID (`ENSG00000117054`) or a gene symbol (`ACADM`).
The XML is cached locally, and the HPA release is recorded in every row.

In [ ]:
gene = hxp.fetch_gene("ACADM")
gene.metadata()

In [ ]:
samples = gene.samples  # one row per sample (long format)
samples.groupby("assay_type").size()

In [ ]:
brain = hxp.to_frame(gene, assays=["humanBrain"])
brain.head()

## 3. Check against HPA's own aggregates

For every tissue or brain subregion, the mean of the per-sample nTPM should equal HPA's
`normalizedRNAExpression` value (both are rounded to one decimal, so differences of up to
0.1 nTPM are expected).

In [ ]:
keys = ["assay_type", "organ", "region"]
means = gene.samples.groupby(keys)["value"].mean().rename("mean_of_samples")
agg = gene.aggregates.query("level_type == 'normalizedRNAExpression' and n_samples > 0")
check = agg.set_index(keys)["value"].rename("hpa_ntpm").to_frame().join(means)
check["abs_diff"] = (check["mean_of_samples"] - check["hpa_ntpm"]).abs()
print("blocks compared:", len(check))
print("max |difference| (nTPM):", round(check["abs_diff"].max(), 3))

## 4. Plot human brain samples by region

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

order = list(brain.groupby("organ")["value"].median().sort_values().index)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.boxplot([np.log10(brain.loc[brain["organ"] == o, "value"] + 1) for o in order])
ax.set_xticks(range(1, len(order) + 1))
ax.set_xticklabels(order)
ax.set_ylabel("log10(nTPM + 1)")
ax.set_title(f"{gene.gene_name}, humanBrain (HPA v{gene.hpa_version})")
plt.tight_layout()

## 5. Several genes and export

`export` writes one Excel sheet per assay, an `aggregates` sheet and a `metadata` sheet
with the HPA/Ensembl versions and the download date. Use `.csv`, `.tsv` or `.parquet` for
flat files.

In [ ]:
genes = hxp.fetch_genes(["ACADM", "GFAP", "ACTB"])
hxp.export(genes, "hpa_results.xlsx")

In [ ]:
try:  # offer the file for download when running in Google Colab
    from google.colab import files

    files.download("hpa_results.xlsx")
except ImportError:
    print("Saved hpa_results.xlsx")

## 6. Parse an XML file you already have (optional)

Download `https://www.proteinatlas.org/<ENSG>.xml` yourself, upload it to Colab, and parse
it. Both `.xml` and `.xml.gz` files are accepted.

In [ ]:
# gene = hxp.parse_xml("ENSG00000117054.xml")